# <span style="font-family:cursive;text-align:center">⬇️ Import Libraries</span>

In [1]:
import numpy as np
import pandas as pd
import glob
import os
from datasets import Dataset
import torch
# Set the display options for pandas
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', None)

In [2]:
# Set the device to cuda if available, otherwise cpu
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# <span style="font-family:cursive;text-align:center">⬇️ Import Data</span>

In [3]:
# Load the data from a tsv file
data = pd.read_csv("/kaggle/input/demo-data/dataset/train/boxes_transcripts_labels/004a4c67-561d-4d9c-9ef2-47cb15fbdf08_document-3_page-1.tsv", header = None)
# Assign column names to the data
data.columns = ['start_index', 'end_index', 'x_top_left', 'y_top_left', 'x_bottom_right', 'y_bottom_right', 'transcript', 'field']

In [4]:
data.sample(100)

,start_index,end_index,x_top_left,y_top_left,x_bottom_right,y_bottom_right,transcript,field
307,8199,8203,889,971,919,992,other,OTHER
227,6499,6503,141,878,188,903,B--To,OTHER
88,2471,2475,139,290,214,320,Ariel,employerAddressStreet_name
106,3160,3161,1012,463,1024,477,12,OTHER
435,12565,12569,575,1516,617,1542,cages,OTHER
333,9263,9266,1174,1090,1203,1110,ties,OTHER
399,11280,11281,1017,1333,1030,1350,to,OTHER
350,9685,9690,330,1143,413,1175,Courts,OTHER
39,969,970,856,960,871,978,on,OTHER
243,6603,6605,369,876,396,896,the,OTHER


In [5]:
data['field'].unique()

array(['OTHER', 'box2FederalIncomeTaxWithheld', 'ssnOfEmployee',
       'box1WagesTipsAndOtherCompensations',
       'box4SocialSecurityTaxWithheld', 'box3SocialSecurityWages',
       'einEmployerIdentificationNumber', 'employerName',
       'employerAddressStreet_name', 'employerAddressState',
       'employerAddressZip', 'employerAddressCity', 'employeeName',
       'taxYear', 'box17StateIncomeTax', 'box16StateWagesTips'],
      dtype=object)

In [6]:
# Define a dictionary of ner labels and their corresponding numeric codes
ner_labels = { 0 : "OTHER",
1 : "employerName",
2 : "employerAddressStreet_name",
3 : "employerAddressCity",
4 : "employerAddressState",
5 : "employerAddressZip",
6 : "einEmployerIdentificationNumber",
7 : "employeeName",
8 : "ssnOfEmployee",
9 : "box1WagesTipsAndOtherCompensations",
10 : "box2FederalIncomeTaxWithheld",
11 : "box3SocialSecurityWages",
12 : "box4SocialSecurityTaxWithheld",
13 : "box16StateWagesTips",
14 : "box17StateIncomeTax",
15 : "taxYear"
}

In [7]:
# Create a reverse dictionary of ner labels and their numeric codes
ner_labels_dict = {}
for key in ner_labels.keys():
    ner_labels_dict.update({ner_labels[key] : key})

In [8]:
ner_labels_dict

{'OTHER': 0,
 'employerName': 1,
 'employerAddressStreet_name': 2,
 'employerAddressCity': 3,
 'employerAddressState': 4,
 'employerAddressZip': 5,
 'einEmployerIdentificationNumber': 6,
 'employeeName': 7,
 'ssnOfEmployee': 8,
 'box1WagesTipsAndOtherCompensations': 9,
 'box2FederalIncomeTaxWithheld': 10,
 'box3SocialSecurityWages': 11,
 'box4SocialSecurityTaxWithheld': 12,
 'box16StateWagesTips': 13,
 'box17StateIncomeTax': 14,
 'taxYear': 15}

In [9]:
# Create a new column in the data with the numeric codes for the ner tags
data['ner_tags'] = data['field'].map(lambda x: ner_labels_dict[x])

In [10]:
# Convert the ner tags column to a list
data['ner_tags'].to_list()

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 10,
 10,
 0,
 0,
 0,
 0,
 0,
 8,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 9,
 9,
 9,
 0,
 0,
 0,
 12,
 12,
 11,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 6,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 2,
 2,
 2,
 4,
 5,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 3,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 7,
 7,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 15,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0

In [11]:
# Convert the transcript column to a list
data['transcript'].to_list()

['12a',
 'being',
 'This',
 'information',
 'compensation',
 'Visit',
 'the',
 'IRS',
 'Website',
 'Safe,',
 'Accurate,',
 're',
 'file',
 'a',
 'a',
 'www.irs.goviefile',
 'FAST!',
 'Use',
 'Federal',
 'income',
 'tax',
 'withheld',
 'OMB',
 'No.',
 '1545-0008',
 '24033.',
 '1',
 'a',
 "Employee's",
 'social',
 'security',
 'number',
 '380-62-5767',
 'Wages,',
 'tips,',
 'other',
 'c',
 'e',
 'at',
 'on',
 'Social',
 'security',
 'tax',
 'withheld',
 '80470',
 '.',
 '88',
 'Social',
 'security',
 'wages',
 '5710',
 '.23',
 '74643.59',
 'Medicare',
 'tax',
 'withheld',
 'b',
 'Employer',
 '48-3726648',
 'identification',
 'number',
 '(EIN)',
 '2868.23',
 'Medicare',
 'wages',
 'and',
 'tips',
 '98904',
 '.',
 '37',
 'Allocated',
 '98904.',
 'tips',
 '37',
 'c',
 "Employer's",
 'name,',
 'address,',
 'and',
 'Flores-Robinson',
 'and',
 'Sons',
 'ZIP',
 'code',
 'Social',
 'security',
 'tips',
 '364',
 'Ariel',
 'Courts',
 'AK',
 '12316-9561',
 '74643.59',
 '10',
 'Dependent',
 'care',
 

In [12]:
# Get the unique values of the ner tags column
data['ner_tags'].unique()

array([ 0, 10,  8,  9, 12, 11,  6,  1,  2,  4,  5,  3,  7, 15, 14, 13])

In [13]:
# Define a function to load data from a given path
def load_data(path):
    directory = path
    # Get all the tsv files in the directory
    tsv_file = glob.glob(directory + '/*.tsv')
    # Create an empty dataframe with two columns: transcript and ner_tags
    data = pd.DataFrame(columns = ['transcript', 'ner_tags'])
    # Loop through each tsv file in the directory
    for filename in tsv_file:
        # Read the tsv file as a dataframe
        data_i = pd.read_csv(filename, header=None)
        # Assign column names to the dataframe
        data_i.columns = ['start_index', 'end_index', 'x_top_left', 'y_top_left', 'x_bottom_right', 'y_bottom_right', 'transcript', 'field']
        # Drop any rows that have missing values in the transcript column
        data_i.dropna(subset=['transcript'], inplace=True)
        # Convert the transcript column to a list
        transcript = data_i['transcript'].to_list()
        # Map the field column to the numeric codes using the reverse dictionary
        data_i['field'] = data_i['field'].map(lambda x: ner_labels_dict[x])
        # Convert the field column to a list of ner tags
        ner_labels = data_i['field'].to_list()
        # Get the length of the transcript list
        transcript_len = len(transcript)
        
        # If the transcript list is longer than 300, split it into two parts and append them as separate rows in the data dataframe
        if transcript_len > 300:
            transcript_1 = transcript[:(transcript_len//2)]
            ner_labels_1 = ner_labels[:(transcript_len//2)]
            transcript_2 = transcript[(transcript_len//2):]
            ner_labels_2 = ner_labels[(transcript_len//2):]
            data.loc[len(data)] = [transcript_1, ner_labels_1]
            data.loc[len(data)] = [transcript_2, ner_labels_2]
        # Otherwise, append the transcript list and the ner tags list as a single row in the data dataframe    
        else:
            data.loc[len(data)] = [transcript, ner_labels]
    # Return the data dataframe
    return data

In [14]:
%%time
# Measure the execution time of loading the train and validation data
train_data = load_data('/kaggle/input/demo-data/dataset/train/boxes_transcripts_labels')
val_data = load_data('/kaggle/input/demo-data/dataset/val_w_ann/boxes_transcripts_labels')

CPU times: user 4.04 s, sys: 105 ms, total: 4.15 s
Wall time: 6.26 s


In [15]:
train_data.loc[0]

transcript    [(C-Box, Profile., Information,, Medicare, :46, 88415, Employer's, 1653.98, Employee, Reference, W-2, Wage, !, ), and, Tax, Copy, 2018, W-2, and, EARNINGS, SUMMARY, Stateme, Copy, C, for, employee...
ner_tags      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...
Name: 0, dtype: object

In [16]:
val_data.head()

,transcript,ner_tags
0,"[1, Wages,, tips,, other, comp., 2, Federal, income, tax, withheld, 76710.3, 11444.41, 3, Social, security, wages, 4, Social, security, tax, withheld, 90562.66, 6928.04, 5, Medicare, wages, and, t...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 4, 5, 0, 0, ..."
1,"[a, Employee's, social, security, number, Safe,, Accurate., Visit, the, IRS, Website, Re, ~, file, 071-51-2027, OMB, No., 1545-0008, FAST!, US, at, www.irs.gov/efile., b, Employer, identification,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 9, 9, 10, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 11, 11, 11, 12, 12, 12..."
2,"[Information,, 2018AOP., ex, 224720.73, Employee, Reference, W-2, and, Tax, 2018, W-2, and, EARNINGS, SUMMARY, 2018, This, blue, Earnings, Summary, section, is, included, with, your, W-2, to, help...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0,..."
3,"[17, State, inco, STATE:, 6535.39, 142808., etc., 62, 19, Local, income, tax, Local, wages,, tips,, Locality, name, Wages,, tips,, other, ., 2018, LLC, Social, sexy, me, tax, win, 209331, -92, led...","[0, 0, 0, 0, 14, 13, 0, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[23838.77, Medicare, tax, 2018, W-2, and, EARNINGS, SUMMARY, ADP, Employee, Reference, Copy, his, blue, Earnings, Summary, section, is, included, with, your, W-2, to, help, describe, portions, in,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


In [17]:
train_data.shape, val_data.shape

((844, 2), (284, 2))

# Transformer preprocessing

In [18]:
# Show the first row of the train data
from transformers import AutoTokenizer
# Load the tokenizer for the distilbert-base-uncased model
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [19]:
# Get the first transcript and ner tags from the train data
example_1 = train_data['transcript'][0]
ner = train_data['ner_tags'][0]

In [20]:
# Tokenize the transcript using the tokenizer
tokens = tokenizer(example_1, is_split_into_words=True, truncation=True)
# Show the tokens
tokens

{'input_ids': [101, 1006, 1039, 1011, 3482, 6337, 1012, 2592, 1010, 27615, 1024, 4805, 6070, 23632, 2629, 11194, 1005, 1055, 29309, 1012, 5818, 7904, 4431, 1059, 1011, 1016, 11897, 999, 1007, 1998, 4171, 6100, 2760, 1059, 1011, 1016, 1998, 16565, 12654, 2110, 4168, 6100, 1039, 2005, 7904, 1005, 1055, 1040, 2491, 2193, 29466, 1012, 13058, 1012, 2023, 2630, 16565, 12654, 2930, 2003, 2443, 2007, 2115, 1059, 1011, 1016, 2000, 2393, 6235, 8810, 1999, 2062, 6987, 2760, 1996, 7901, 2217, 2950, 2236, 2592, 2008, 2017, 2089, 2036, 2424, 14044, 1012, 29466, 5018, 18168, 2497, 2053, 1012, 16666, 2629, 1011, 2199, 2620, 11194, 1005, 1055, 2171, 1010, 4769, 1010, 1998, 14101, 3642, 11194, 2224, 2069, 8685, 1011, 3817, 15492, 4748, 2361, 1015, 1012, 1996, 2206, 2592, 11138, 2115, 2345, 2760, 3477, 24646, 2497, 4606, 2151, 24081, 7864, 2011, 2115, 11194, 1012, 21064, 2581, 2575, 5655, 3137, 2225, 17217, 2139, 6070, 23632, 2629, 1011, 5641, 2620, 2487, 7977, 3477, 7886, 22907, 2581, 1019, 2591, 3036, 

In [21]:
# Define a function to tokenize a given data
def tokenize_func(data):
    # Return the tokenized transcript using the tokenizer
    return tokenizer(data['transcript'], is_split_into_words=True, truncation=True)

In [22]:
# Define a function to tokenize a given data
def tokenize_data(data):
    # Convert the data to a Dataset object from the datasets library
    data = Dataset.from_pandas(data)
    # Apply the tokenize function to the data and map it to a new column
    data = data.map(tokenize_func)
    # Remove the index column from the data
    data = data.remove_columns(['__index_level_0__'])
    # Return the data as a pandas dataframe
    return data.to_pandas()

In [23]:
# Get the transcript column from the train data
train_data['transcript']

0      [(C-Box, Profile., Information,, Medicare, :46, 88415, Employer's, 1653.98, Employee, Reference, W-2, Wage, !, ), and, Tax, Copy, 2018, W-2, and, EARNINGS, SUMMARY, Stateme, Copy, C, for, employee...
1      [Dept., 2, Federal, income, tax, withheld, Employer's, name,, address, ,, and, ZIP, code, security, wages, Lopez-Daniel, PLC, Employer, use, only, Medicare, wages, and, 4, Social, security, tax, w...
2      [Wages,, tips,, other, comp., 2, Federal, income, tax, withheld, 240330, ., 89, 35494, ., 65, 3, Social, security, wages, 4, Social, security, tax, withheld, 280634.5, 21468, .54, 5, Medicare, wag...
3      [1, Wages,, tips,, other, comp., 2, Federal, income, tax, withheld, 233146.9, 73449.38, 3, Social, security, wages, 4, Social, security, tax, withheld, 208770.54, 15970.95, 5, Medicare, wages, and...
4      [23593.7, 18732.51, 245653.79, name,, address,, W-2, 2018, W-2, and, EARNINGS, SUMMARY, ADP, Employee, Reference, Copy, his, blue, Earnings, Summary, section, is, in

In [24]:
# Tokenize the train and validation data and convert them to pandas dataframes
train = tokenize_data(train_data)
val = tokenize_data(val_data)

  0%|          | 0/844 [00:00<?, ?ex/s]

  0%|          | 0/284 [00:00<?, ?ex/s]

In [25]:
train.head()

,transcript,ner_tags,input_ids,attention_mask
0,"[(C-Box, Profile., Information,, Medicare, :46, 88415, Employer's, 1653.98, Employee, Reference, W-2, Wage, !, ), and, Tax, Copy, 2018, W-2, and, EARNINGS, SUMMARY, Stateme, Copy, C, for, employee...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[101, 1006, 1039, 1011, 3482, 6337, 1012, 2592, 1010, 27615, 1024, 4805, 6070, 23632, 2629, 11194, 1005, 1055, 29309, 1012, 5818, 7904, 4431, 1059, 1011, 1016, 11897, 999, 1007, 1998, 4171, 6100, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,"[Dept., 2, Federal, income, tax, withheld, Employer's, name,, address, ,, and, ZIP, code, security, wages, Lopez-Daniel, PLC, Employer, use, only, Medicare, wages, and, 4, Social, security, tax, w...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[101, 29466, 1012, 1016, 2976, 3318, 4171, 2007, 24850, 11194, 1005, 1055, 2171, 1010, 4769, 1010, 1998, 14101, 3642, 3036, 12678, 8685, 1011, 3817, 15492, 11194, 2224, 2069, 27615, 12678, 1998, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,"[Wages,, tips,, other, comp., 2, Federal, income, tax, withheld, 240330, ., 89, 35494, ., 65, 3, Social, security, wages, 4, Social, security, tax, withheld, 280634.5, 21468, .54, 5, Medicare, wag...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 9, 10, 10, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, ...","[101, 12678, 1010, 10247, 1010, 2060, 4012, 2361, 1012, 1016, 2976, 3318, 4171, 2007, 24850, 11212, 22394, 2692, 1012, 6486, 27878, 2683, 2549, 1012, 3515, 1017, 2591, 3036, 12678, 1018, 2591, 303...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,"[1, Wages,, tips,, other, comp., 2, Federal, income, tax, withheld, 233146.9, 73449.38, 3, Social, security, wages, 4, Social, security, tax, withheld, 208770.54, 15970.95, 5, Medicare, wages, and...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 4, ...","[101, 1015, 12678, 1010, 10247, 1010, 2060, 4012, 2361, 1012, 1016, 2976, 3318, 4171, 2007, 24850, 22115, 16932, 2575, 1012, 1023, 6421, 22932, 2683, 1012, 4229, 1017, 2591, 3036, 12678, 1018, 259...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,"[23593.7, 18732.51, 245653.79, name,, address,, W-2, 2018, W-2, and, EARNINGS, SUMMARY, ADP, Employee, Reference, Copy, his, blue, Earnings, Summary, section, is, included, with, your, W-2, to, he...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[101, 17825, 2683, 2509, 1012, 1021, 7612, 2475, 1012, 4868, 21005, 26187, 2509, 1012, 6535, 2171, 1010, 4769, 1010, 1059, 1011, 1016, 2760, 1059, 1011, 1016, 1998, 16565, 12654, 4748, 2361, 7904,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [26]:
# Define a function to create ner tags for each token in a given transcript and ner tags list
def create_ner_tags(tokens, ner_tags):
    # Preprocess the tokens using the tokenizer
    preprocessed_tokens = tokenizer(tokens, is_split_into_words=True, truncation=True)
    # Get the input ids and word ids from the preprocessed tokens
    input_ids = preprocessed_tokens['input_ids']
    word_ids = preprocessed_tokens.word_ids()
    
    # Create an empty list for storing the ner tags for each token
    ner_tags_for_tokens = []
    # Loop through each token id in the input ids list
    for ind, token in enumerate(input_ids):
        # If the word id is None, append -100 to the ner tags list (this means that this token will be ignored for loss calculation)
        if word_ids[ind] == None:
            ner_tags_for_tokens.append(-100)
        # If the word id is equal to the previous word id, append -100 to the ner tags list (this means that this token is part of a subword and will be ignored for loss calculation)
        elif word_ids[ind] == word_ids[ind - 1]:
            ner_tags_for_tokens.append(-100)
        # Otherwise, append the corresponding ner tag from the ner tags list (this means that this token is a whole word and will be used for loss calculation)
        else:
            ner_tags_for_tokens.append(ner_tags[word_ids[ind]])
    # Return the ner tags for tokens list
    return ner_tags_for_tokens

In [27]:
# Create ner tags for each token in the example transcript and ner tags list
tags = create_ner_tags(example_1, ner)

In [28]:
# Loop through each token id and the corresponding ner tag in the input ids and tags lists
for ind, tok in enumerate(tokenizer.convert_ids_to_tokens(tokens['input_ids'])):
    print(tok, tags[ind])

[CLS] -100
( 0
c -100
- -100
box -100
profile 0
. -100
information 0
, -100
medicare 0
: 0
46 -100
88 0
##41 -100
##5 -100
employer 0
' -100
s -100
1653 0
. -100
98 -100
employee 0
reference 0
w 0
- -100
2 -100
wage 0
! 0
) 0
and 0
tax 0
copy 0
2018 0
w 0
- -100
2 -100
and 0
earnings 0
summary 0
state 0
##me -100
copy 0
c 0
for 0
employee 0
' -100
s -100
d 0
control 0
number 0
dept 0
. -100
corp 0
. -100
this 0
blue 0
earnings 0
summary 0
section 0
is 0
included 0
with 0
your 0
w 0
- -100
2 -100
to 0
help 0
describe 0
portions 0
in 0
more 0
detail 0
2018 15
the 0
reverse 0
side 0
includes 0
general 0
information 0
that 0
you 0
may 0
also 0
find 0
helpful 0
. -100
dept 0
150 0
om 0
##b -100
no 0
. -100
154 0
##5 -100
- -100
000 -100
##8 -100
employer 0
' -100
s -100
name 0
, -100
address 0
, -100
and 0
zip 0
code 0
employer 0
use 0
only 0
lopez 1
- -100
daniel -100
plc 1
ad 0
##p -100
1 0
. -100
the 0
following 0
information 0
reflects 0
your 0
final 0
2018 0
pay 0
stu 0
##b -100
plus 0

## Creating ner preprocesssed column

In [29]:
%%time
# Measure the execution time of creating ner tags for the train and validation data
train['ner_tags'] = train_data.apply(lambda x: create_ner_tags(x['transcript'], x['ner_tags']), axis=1)

CPU times: user 2.15 s, sys: 1.14 ms, total: 2.15 s
Wall time: 2.15 s


In [30]:
# Apply the create_ner_tags function to the validation data and store the result as a new column
val['ner_tags'] = val_data.apply(lambda x: create_ner_tags(x['transcript'], x['ner_tags']), axis=1)

In [31]:
# Drop the transcript column from the train and validation data
train.drop('transcript', axis=1, inplace=True)
val.drop('transcript', axis=1, inplace=True)

In [32]:
# Convert the train and validation data to Dataset objects from the datasets library
train = Dataset.from_pandas(train)
train = train.rename_column("ner_tags", "labels")
# Rename the ner_tags column to labels in both datasets
val = Dataset.from_pandas(val)
val = val.rename_column("ner_tags", "labels")

In [33]:
# Set the format of both datasets to torch
train.set_format('torch')
val.set_format('torch')

In [34]:
train

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 844
})

In [35]:
# Import the DataCollatorForTokenClassification class from the transformers library
from transformers import DataCollatorForTokenClassification
# Create a data collator object using the tokenizer
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [36]:
!pip install evaluate seqeval

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16165 sha256=78e927a5328844f8428d40b7a05ea65fe4dacae753e8c4910fd649c43fbdedfb
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


In [37]:
# Load the seqeval evaluation metric from the evaluate module
import evaluate

seqeval = evaluate.load("seqeval")

In [38]:
# Import the load_metric function from the datasets library
from datasets import load_metric
metric = load_metric("seqeval")

In [39]:
# labels = [ner_labels[i] for i in train["ner_tags"]]
# metric.compute(predictions=[labels], references=[labels])

In [40]:
ner_labels_dict

{'OTHER': 0,
 'employerName': 1,
 'employerAddressStreet_name': 2,
 'employerAddressCity': 3,
 'employerAddressState': 4,
 'employerAddressZip': 5,
 'einEmployerIdentificationNumber': 6,
 'employeeName': 7,
 'ssnOfEmployee': 8,
 'box1WagesTipsAndOtherCompensations': 9,
 'box2FederalIncomeTaxWithheld': 10,
 'box3SocialSecurityWages': 11,
 'box4SocialSecurityTaxWithheld': 12,
 'box16StateWagesTips': 13,
 'box17StateIncomeTax': 14,
 'taxYear': 15}

In [41]:
# Get a list of ner labels from the dictionary keys
list(ner_labels.keys())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

In [42]:
# Define a dictionary of id to label mapping for each numeric code
id2label = {
    0 : "OTHER",
    1 : "employerName",
    2 : "employerAddressStreet_name",
    3 : "employerAddressCity",
    4 : "employerAddressState",
    5 : "employerAddressZip",
    6 : "einEmployerIdentificationNumber",
    7 : "employeeName",
    8 : "ssnOfEmployee",
    9 : "box1WagesTipsAndOtherCompensations",
    10 : "box2FederalIncomeTaxWithheld",
    11 : "box3SocialSecurityWages",
    12 : "box4SocialSecurityTaxWithheld",
    13 : "box16StateWagesTips",
    14 : "box17StateIncomeTax",
    15 : "taxYear"
}
# Define a dictionary of label to id mapping for each ner label
label2id = {
    'OTHER': 0,
     'employerName': 1,
     'employerAddressStreet_name': 2,
     'employerAddressCity': 3,
     'employerAddressState': 4,
     'employerAddressZip': 5,
     'einEmployerIdentificationNumber': 6,
     'employeeName': 7,
     'ssnOfEmployee': 8,
     'box1WagesTipsAndOtherCompensations': 9,
     'box2FederalIncomeTaxWithheld': 10,
     'box3SocialSecurityWages': 11,
     'box4SocialSecurityTaxWithheld': 12,
     'box16StateWagesTips': 13,
     'box17StateIncomeTax': 14,
     'taxYear': 15
}

# <span style="font-family:cursive;text-align:center">⬇️ Training</span>

In [43]:
# Import the AutoModelForTokenClassification class from the transformers library
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
# Load a model for token classification from a pretrained model name, specifying the number of labels and the id2label and label2id dictionaries
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=16, id2label=id2label, label2id=label2id
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [44]:
# Move the model to the device (cuda or cpu)
model.to(device)

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
    

In [45]:
# Create a training arguments object with various hyperparameters and settings for training and evaluation
training_args = TrainingArguments(
    output_dir="/kaggle/working/",
    learning_rate=2e-5,
    logging_strategy = 'epoch',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to = 'none'
)

In [46]:
# Create a trainer object with the model, the training arguments, the train and validation datasets, the tokenizer, the data collator, and the metric
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=val,
    tokenizer=tokenizer,
    data_collator=data_collator,
#     compute_metrics=metric,
)

In [47]:
# Train the model using the trainer object
trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss
1,0.586400,0.315407
2,0.235300,0.147397
3,0.120400,0.074157
4,0.079400,0.052940
5,0.061300,0.044138
6,0.050600,0.038991
7,0.044500,0.033732
8,0.038500,0.032499
9,0.035700,0.031005
10,0.031900,0.029161


TrainOutput(global_step=1060, training_loss=0.07571141584864202, metrics={'train_runtime': 555.8961, 'train_samples_per_second': 30.365, 'train_steps_per_second': 1.907, 'total_flos': 2200921577092224.0, 'train_loss': 0.07571141584864202, 'epoch': 20.0})

# <span style="font-family:cursive;text-align:center">⬇️ Inference</span>

In [48]:
from transformers import pipeline
# Create a classifier object using the ner pipeline, the trained model, the tokenizer, and the device
classifier = pipeline("ner", model=model, tokenizer=tokenizer, device=device)
# Set the tokenizer attribute is_split_into_words to True (this means that the input is already split into words)
classifier.tokenizer.is_split_into_words = True

In [49]:
# Apply the classifier to a list of words
tokens = classifier(["77796.34", "3759.51" ,"withheby" ,"2018", "W-2", "and" ,"EARNINGS", "SUMMARY", "ADP", "Employee"])
# Create an empty list for storing the labels
labels = []
# Loop through each element in the tokens list
for li in tokens:
    # Loop through each dictionary in the element
    for dic in li:
        # Append the entity value from the dictionary to the labels list
        labels.append(dic['entity'])

In [50]:
labels

['box3SocialSecurityWages',
 'box3SocialSecurityWages',
 'box3SocialSecurityWages',
 'box3SocialSecurityWages',
 'box3SocialSecurityWages',
 'box3SocialSecurityWages',
 'box4SocialSecurityTaxWithheld',
 'box3SocialSecurityWages',
 'box3SocialSecurityWages',
 'box4SocialSecurityTaxWithheld',
 'OTHER',
 'employerAddressCity',
 'OTHER',
 'taxYear',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER']

In [51]:
tokens

[[{'entity': 'box3SocialSecurityWages',
   'score': 0.34892526,
   'index': 1,
   'word': '77',
   'start': 0,
   'end': 2},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.5969268,
   'index': 2,
   'word': '##7',
   'start': 2,
   'end': 3},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.5498678,
   'index': 3,
   'word': '##9',
   'start': 3,
   'end': 4},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.6177936,
   'index': 4,
   'word': '##6',
   'start': 4,
   'end': 5},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.5117226,
   'index': 5,
   'word': '.',
   'start': 5,
   'end': 6},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.4532416,
   'index': 6,
   'word': '34',
   'start': 6,
   'end': 8}],
 [{'entity': 'box4SocialSecurityTaxWithheld',
   'score': 0.2601858,
   'index': 1,
   'word': '375',
   'start': 0,
   'end': 3},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.469249,
   'index': 2,
   'word': '##9',
   'start': 3,
   'end': 4

In [52]:
# Define a function to extract labels from inference output
def extract_labels_from_inference(opt):
    lab = []
    for li in opt:
        lab.append(li[0]['entity'])
    return lab

In [53]:
# Apply the function to the tokens list
extract_labels_from_inference(tokens)

['box3SocialSecurityWages',
 'box4SocialSecurityTaxWithheld',
 'OTHER',
 'taxYear',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER']

In [54]:
words = "77796.34 3759.51 withheby 2018 W-2 and EARNINGS SUMMARY ADP Employee"
words = words.split()

In [55]:
len(words)

10

In [56]:
len(labels)

23

In [57]:
tokens

[[{'entity': 'box3SocialSecurityWages',
   'score': 0.34892526,
   'index': 1,
   'word': '77',
   'start': 0,
   'end': 2},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.5969268,
   'index': 2,
   'word': '##7',
   'start': 2,
   'end': 3},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.5498678,
   'index': 3,
   'word': '##9',
   'start': 3,
   'end': 4},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.6177936,
   'index': 4,
   'word': '##6',
   'start': 4,
   'end': 5},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.5117226,
   'index': 5,
   'word': '.',
   'start': 5,
   'end': 6},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.4532416,
   'index': 6,
   'word': '34',
   'start': 6,
   'end': 8}],
 [{'entity': 'box4SocialSecurityTaxWithheld',
   'score': 0.2601858,
   'index': 1,
   'word': '375',
   'start': 0,
   'end': 3},
  {'entity': 'box3SocialSecurityWages',
   'score': 0.469249,
   'index': 2,
   'word': '##9',
   'start': 3,
   'end': 4

In [58]:
# Preprocess the words using the tokenizer (without adding special tokens)
preprocessed_tokens = tokenizer(words, is_split_into_words=True, truncation=True, add_special_tokens = False)

In [59]:
preprocessed_tokens

{'input_ids': [6255, 2581, 2683, 2575, 1012, 4090, 18034, 2683, 1012, 4868, 2007, 5369, 3762, 2760, 1059, 1011, 1016, 1998, 16565, 12654, 4748, 2361, 7904], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## Inference on val data

In [60]:
# Define a function to load validation data from a given path
def load_val_data(path):
    directory = path

    tsv_file = glob.glob(directory + '/*.tsv')

    data = pd.DataFrame(columns = ['transcript'])

    for filename in tsv_file:
        data_i = pd.read_csv(filename, header=None)
        data_i.columns = ['start_index', 'end_index', 'x_top_left', 'y_top_left', 'x_bottom_right', 'y_bottom_right', 'transcript']
        data_i.dropna(subset=['transcript'], inplace=True)
        
        transcript = data_i['transcript'].to_list()
        
        data.loc[len(data)] = [transcript]
        
    return data

In [61]:
# Load the validation data without labels from a given path
val_data_without_labels = load_val_data('/kaggle/input/demo-data/dataset/val/boxes_transcripts')

In [62]:
# Get the third transcript from the validation data without labels
tok = val_data_without_labels['transcript'][2]

In [63]:
tok, len(tok)

(['Information,',
  '2018AOP.',
  'ex',
  '224720.73',
  'Employee',
  'Reference',
  'W-2',
  'and',
  'Tax',
  '2018',
  'W-2',
  'and',
  'EARNINGS',
  'SUMMARY',
  '2018',
  'This',
  'blue',
  'Earnings',
  'Summary',
  'section',
  'is',
  'included',
  'with',
  'your',
  'W-2',
  'to',
  'help',
  'describe',
  'portions',
  'in',
  'more',
  'detail',
  '3374305',
  'The',
  'reverse',
  'side',
  'includes',
  'general',
  'information',
  'that',
  'you',
  'may',
  'also',
  'find',
  'helpful.',
  'DUMB',
  'Dis.',
  '1565-609',
  'Salazar-Nguyen',
  'Lit',
  'Employer',
  'use',
  'only',
  'guyen',
  '227',
  'Allison',
  'Skyway',
  'code',
  'Suite',
  '370',
  'ADP',
  '1.',
  'The',
  'following',
  'information',
  'reflects',
  'your',
  'final',
  '2',
  '1',
  'pay',
  'stub',
  'plus',
  'any',
  'adjustments',
  'submitted',
  'by',
  'your',
  'employer.',
  'North',
  'Stephen',
  'No',
  'Gross',
  'Pay',
  '80044-4080',
  '208',
  '542',
  '1',
  'Social',


In [64]:
# Define a function to predict the ner tags for a given transcript
def predict_val(example):
    ner_tags = []
    len_example = len(example)
    if len_example > 300:
        
        example_1 = example[:(len_example//2)]
        example_2 = example[(len_example//2):]
        
        lab_1 = classifier(example_1)
        lab_2 = classifier(example_2)
        
        lab_1 = extract_labels_from_inference(lab_1)
        lab_2 = extract_labels_from_inference(lab_2)
        
        ner_tags += lab_1
        ner_tags += lab_2
    
    else:
        
        lab = classifier(example)
        lab = extract_labels_from_inference(lab)
        ner_tags += lab
    
    return ner_tags

In [65]:
# Get all the tsv files in a given directory

tsv_file = glob.glob('/kaggle/working/val/boxes_transcripts' + '/*.tsv')

In [66]:
len(tsv_file)

0

In [67]:
import os
# Define a function to generate validation tsv files with predicted labels
def generate_val_tsvs(path):
    
    if not os.path.exists('val'):
        os.mkdir('val')
        os.mkdir('val/boxes_transcripts')
    
    directory = path

    tsv_file = glob.glob(directory + '/*.tsv')

    for filename in tsv_file:
        data_i = pd.read_csv(filename, header=None)
        data_i.columns = ['start_index', 'end_index', 'x_top_left', 'y_top_left', 'x_bottom_right', 'y_bottom_right', 'transcript']
        data_i.dropna(subset=['transcript'], inplace=True)
        
        transcript = data_i['transcript'].to_list()
        
        pred = predict_val(transcript)
        
        data_i['field'] = pred
        
        data_i.to_csv(f"val/boxes_transcripts/{filename.split('/')[-1]}", index=False, header=None)

In [68]:
%%time
# Measure the execution time of generating the validation tsv files with labels
generate_val_tsvs('/kaggle/input/demo-data/dataset/val/boxes_transcripts')

/opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentiall

CPU times: user 7min 29s, sys: 571 ms, total: 7min 29s
Wall time: 7min 31s


# <span style="font-family:cursive;text-align:center">⬇️ Generating metrics.tsv</span>

In [69]:
import os
import csv
import pandas as pd

'''
Entities:
1. employerName
2. employerAddressStreet_name
3. employerAddressCity
4. employerAddressState
5. employerAddressZip
6. einEmployerIdentificationNumber
7. employeeName
8. ssnOfEmployee
9. box1WagesTipsAndOtherCompensations
10. box2FederalIncomeTaxWithheld
11. box3SocialSecurityWages
12. box4SocialSecurityTaxWithheld
13. box16StateWagesTips
14. box17StateIncomeTax
15. taxYear
'''



'''
Description: The fuction yields the standard precision, recall and f1 score metrics

arguments:
    TP -> int
    FP -> int
    FN -> int

returns: float, float, float
'''
def performance(TP, FP, FN):
    
    if (TP+FP) == 0:
        precision = 0
    else:
        precision = TP/float((TP+FP))
        
    if (TP+FN) == 0:
        recall = 0
    else:
        recall = TP/float((TP+FN))
    
    if (recall!= 0) and (precision!= 0):
        f1_score = (2.0*precision*recall)/(precision+recall)
    else:
        f1_score = 0
    
    return precision, recall, f1_score
    
    
    
    
'''
Description: The fuction yields a dataframe containing entity-wise performance metrics

arguments:
    true_labels -> list
    pred_labels -> lisyt
    
returns: pandas dataframe
'''
def get_dataset_metrics(true_labels, pred_labels):
    
    metrics_dict = dict()
    
    for true_label, pred_label in zip(true_labels, pred_labels):
        if true_label not in metrics_dict:
            metrics_dict[true_label] = {"TP":0, "FP":0, "FN":0, "Support":0}
        
        if true_label != "OTHER":
            metrics_dict[true_label]["Support"] += 1
            
            if true_label == pred_label:
                metrics_dict[true_label]["TP"] += 1
            
            elif pred_label == "OTHER":
                metrics_dict[true_label]["FN"] += 1
            
        else:
            if pred_label != "OTHER":
                metrics_dict[pred_label]["FP"] += 1
           
    df = pd.DataFrame()
    
    for field in metrics_dict:
        precision, recall, f1_score = performance(metrics_dict[field]["TP"], metrics_dict[field]["FP"], metrics_dict[field]["FN"])
        support = metrics_dict[field]["Support"]
        
        if field != "OTHER":
            temp_df = pd.DataFrame([[precision, recall, f1_score, support]], columns=["Precision", "Recall", "F1-Score", "Support"], index=[field])
#             df = df.append(temp_df)
            df = pd.concat([df, temp_df])
    
    return df




'''
Description: The fuction yields a dataframe containing entity-wise performance metrics for a single document
(make sure the doc id is the same)

arguments:
    doc_true -> tsv file with with labels in the last column (8 th column (1-indexed))
    doc_pred -> tsv file with labels in the last column (8 th column (1-indexed)), as predicted by the model
    
returns: list, list
'''
def get_doc_labels(doc_true, doc_pred):

    true_labels = [row[-1] for row in csv.reader(open(doc_true, "r"))]
    pred_labels = [row[-1] for row in csv.reader(open(doc_pred, "r"))]

    return true_labels, pred_labels

'''
Description: The fuction yields a dataframe containing entity-wise performance metrics for all documents
(make sure the doc ids are the same in both the paths)

arguments:
    doc_true -> string (directory containing the ground truth tsv files)
    doc_pred -> string (directory containing the predicted tsv files)
    save -> bool (saves the metrics file in your working directory)
returns: pandas dataframe
'''
def get_dataset_labels(true_path, pred_path, save=False):
    
    y_true, y_pred = [], []
    
    for true_file in os.listdir(true_path):
        for pred_file in os.listdir(pred_path):
            if (".tsv" in true_file) and (".tsv" in pred_file):
                if true_file == pred_file:
                    
                    true_file, pred_file = f"{true_path}/{true_file}", f"{pred_path}/{pred_file}"
                    true_labels, pred_labels = get_doc_labels(true_file, pred_file)
                    
                    y_true.extend(true_labels)
                    y_pred.extend(pred_labels)
            
    df = get_dataset_metrics(y_true, y_pred)
    print(df)
    if save == True:
        df.to_csv("eval_metrics.tsv")



if __name__ == "__main__":
    
    # template to run your own evaluation

    doc_true = f"/kaggle/input/demo-data/dataset/val_w_ann/boxes_transcripts_labels"
    doc_pred = f"/kaggle/working/val/boxes_transcripts"

    get_dataset_labels(doc_true, doc_pred, save=True)

                                    Precision    Recall  F1-Score  Support
box1WagesTipsAndOtherCompensations   0.088235  0.009259  0.016760      359
box2FederalIncomeTaxWithheld         0.000000  0.000000  0.000000      368
box3SocialSecurityWages              0.012324  0.082840  0.021456      353
box4SocialSecurityTaxWithheld        0.012821  0.047761  0.020215      359
employerName                         0.122807  0.011532  0.021084      696
employerAddressStreet_name           0.013468  0.005755  0.008065      786
employerAddressCity                  0.030211  0.034843  0.032362      305
employerAddressState                 0.000000  0.000000  0.000000      196
employerAddressZip                   0.040284  0.094444  0.056478      199
einEmployerIdentificationNumber      0.023969  0.142045  0.041017      195
ssnOfEmployee                        0.027484  0.080745  0.041009      173
employeeName                         0.000000  0.000000  0.000000      404
box16StateWagesTips      

In [70]:
eval_df = pd.read_csv("/kaggle/working/eval_metrics.tsv")
eval_df

,Unnamed: 0,Precision,Recall,F1-Score,Support
0,box1WagesTipsAndOtherCompensations,0.088235,0.009259,0.016760,359
1,box2FederalIncomeTaxWithheld,0.000000,0.000000,0.000000,368
2,box3SocialSecurityWages,0.012324,0.082840,0.021456,353
3,box4SocialSecurityTaxWithheld,0.012821,0.047761,0.020215,359
4,employerName,0.122807,0.011532,0.021084,696
5,employerAddressStreet_name,0.013468,0.005755,0.008065,786
6,employerAddressCity,0.030211,0.034843,0.032362,305
7,employerAddressState,0.000000,0.000000,0.000000,196
8,employerAddressZip,0.040284,0.094444,0.056478,199
9,einEmployerIdentificationNumber,0.023969,0.142045,0.041017,195
